# Hierarchical Cell Type Evaluation with OnClass

This notebook demonstrates how to evaluate cell type predictions using both standard metrics (from cellxgene_v2_mlp.ipynb) and hierarchical metrics via OnClass.

In [1]:
# Setup and imports
%run notebook_setup.ipynb

from dotenv import load_dotenv
load_dotenv()

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.metrics import log_loss, f1_score, precision_score, recall_score
import warnings
warnings.filterwarnings('ignore')

2025-07-11 04:36:51 - autoreload enabled
2025-07-11 04:37:02 - repo_dir set to /Users/rj/personal/GenePT-tools
2025-07-11 04:37:16 - data_dir set to /Users/rj/personal/GenePT-tools/data


File already exists at /Users/rj/personal/GenePT-tools/data/GenePT_emebdding_v2.zip
Extracting files...
Extracting GenePT_emebdding_v2/
Skipping GenePT_emebdding_v2/NCBI_UniProt_summary_of_genes.json - already exists with same size
Skipping GenePT_emebdding_v2/GenePT_gene_embedding_ada_text.pickle - already exists with same size
Skipping GenePT_emebdding_v2/GenePT_gene_protein_embedding_model_3_text.pickle. - already exists with same size
Skipping GenePT_emebdding_v2/NCBI_summary_of_genes.json - already exists with same size
Extraction complete!
Skipping embedding_original_ada_text.parquet - already exists
Skipping embedding_original_large_3.parquet - already exists
Skipping embedding_associations_age_cell_type_drugs_pathways_openai_large.parquet - already exists
Skipping embedding_associations_age_drugs_pathways_openai_large.parquet - already exists
Skipping embedding_associations_cell_type_openai_large.parquet - already exists
Skipping embedding_associations_cell_type_tissue_drug_pat

## Load Cell Ontology

OnClass uses the Cell Ontology for hierarchical evaluation. We'll download it.

In [2]:
# Download Cell Ontology if not already present
import requests
import obonet

cell_ontology_path = data_dir / "cl.obo"
if not cell_ontology_path.exists():
    print("Downloading Cell Ontology...")
    url = "http://purl.obolibrary.org/obo/cl.obo"
    response = requests.get(url)
    with open(cell_ontology_path, 'wb') as f:
        f.write(response.content)
    print("Downloaded!")

# Load the ontology
ontology = obonet.read_obo(cell_ontology_path)
print(f"Loaded Cell Ontology with {len(ontology)} terms")

Loaded Cell Ontology with 17053 terms


## Define Evaluation Functions

These are the same evaluation functions from cellxgene_v2_mlp.ipynb, plus hierarchical metrics.

In [4]:
def mrr_at_k(y_true, y_pred_probs, k):
    """Mean Reciprocal Rank at k"""
    topk_preds = np.argsort(y_pred_probs, axis=1)[:, -k:][:, ::-1]
    rr = []
    for i in range(len(y_true)):
        if y_true[i] in topk_preds[i]:
            rank = np.where(topk_preds[i] == y_true[i])[0][0]
            rr.append(1.0 / (rank + 1))
        else:
            rr.append(0.0)
    return np.mean(rr)

def dcg_at_k(y_true, y_pred_probs, k):
    """Discounted Cumulative Gain at k"""
    topk_preds = np.argsort(y_pred_probs, axis=1)[:, -k:][:, ::-1]
    dcg = []
    for i in range(len(y_true)):
        if y_true[i] in topk_preds[i]:
            rank = np.where(topk_preds[i] == y_true[i])[0][0]
            dcg.append(1.0 / np.log2(rank + 2))
        else:
            dcg.append(0.0)
    return np.mean(dcg)

def recall_at_k(y_true, y_pred_probs, k):
    """Recall at k"""
    topk_preds = np.argsort(y_pred_probs, axis=1)[:, -k:][:, ::-1]
    hits = [y_true[i] in topk_preds[i] for i in range(len(y_true))]
    return np.mean(hits)

def evaluate(model, X, y, batch_size=1024):
    """Standard evaluation from cellxgene_v2_mlp.ipynb"""
    y = np.asarray(y)
    model.eval()
    all_preds = []
    device = next(model.parameters()).device
    
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb = torch.tensor(X[i:i+batch_size], dtype=torch.float32).to(device)
            logits = model(xb)
            preds = torch.softmax(logits, dim=1).cpu().numpy()
            all_preds.append(preds)
    
    all_preds = np.concatenate(all_preds, axis=0)
    y_pred = all_preds.argmax(axis=1)
    
    num_classes = all_preds.shape[1]
    logloss = log_loss(y, all_preds, labels=np.arange(num_classes))
    macro_f1 = f1_score(y, y_pred, average='macro', labels=np.arange(num_classes))
    macro_precision = precision_score(y, y_pred, average='macro', labels=np.arange(num_classes), zero_division=0)
    macro_recall = recall_score(y, y_pred, average='macro', labels=np.arange(num_classes), zero_division=0)

    metrics = {
        "logloss": logloss,
        "macro_f1": macro_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall
    }

    for k in [2, 5, 10]:
        metrics[f"recall_at_{k}"] = recall_at_k(y, all_preds, k)
        metrics[f"mrr_at_{k}"] = mrr_at_k(y, all_preds, k)
        metrics[f"dcg_at_{k}"] = dcg_at_k(y, all_preds, k)

    return metrics, y, all_preds, y_pred

## Hierarchical Evaluation Functions

Now we add the hierarchical evaluation capabilities using OnClass principles.

In [5]:
import networkx as nx
from typing import Set, List, Dict

def build_cell_type_graph(ontology, cell_types):
    """
    Build a graph of cell type relationships from the Cell Ontology.
    """
    # Create directed graph
    G = nx.DiGraph()
    
    # Map cell type names to ontology IDs (this is simplified - real implementation would need proper mapping)
    # For demonstration, we'll create a simple hierarchy
    for node_id, node_data in ontology.nodes(data=True):
        if 'name' in node_data:
            G.add_node(node_data['name'])
    
    # Add edges based on is_a relationships
    for node_id, node_data in ontology.nodes(data=True):
        if 'name' in node_data and 'is_a' in node_data:
            child_name = node_data['name']
            for parent_id in node_data['is_a']:
                if parent_id in ontology:
                    parent_name = ontology.nodes[parent_id].get('name')
                    if parent_name:
                        G.add_edge(parent_name, child_name)
    
    return G

def get_ancestors(node: str, graph: nx.DiGraph) -> Set[str]:
    """Get all ancestors of a node including the node itself"""
    ancestors = {node}
    if node in graph:
        ancestors.update(nx.ancestors(graph, node))
    return ancestors

def calculate_hierarchical_f_score(y_true_labels: List[str], 
                                 y_pred_labels: List[str], 
                                 ontology_graph: nx.DiGraph) -> Dict[str, float]:
    """
    Calculate hierarchical precision, recall, and F-score.
    Based on Kiritchenko et al. (2005).
    """
    total_true_ancestors = 0
    total_pred_ancestors = 0
    total_intersection = 0
    
    for true_label, pred_label in zip(y_true_labels, y_pred_labels):
        true_ancestors = get_ancestors(true_label, ontology_graph)
        pred_ancestors = get_ancestors(pred_label, ontology_graph)
        
        intersection = len(true_ancestors & pred_ancestors)
        
        total_true_ancestors += len(true_ancestors)
        total_pred_ancestors += len(pred_ancestors)
        total_intersection += intersection
    
    # Calculate hierarchical precision and recall
    h_precision = total_intersection / total_pred_ancestors if total_pred_ancestors > 0 else 0
    h_recall = total_intersection / total_true_ancestors if total_true_ancestors > 0 else 0
    
    # Calculate F-score
    if h_precision + h_recall == 0:
        h_f1 = 0
    else:
        h_f1 = 2 * (h_precision * h_recall) / (h_precision + h_recall)
    
    return {
        "hierarchical_precision": h_precision,
        "hierarchical_recall": h_recall,
        "hierarchical_f1": h_f1
    }

def evaluate_with_hierarchy(model, X, y, cell_types, full_count_codes_pdf, ontology_graph, batch_size=1024):
    """
    Evaluate model with both standard and hierarchical metrics.
    """
    # Get standard metrics
    standard_metrics, y_true, all_preds, y_pred = evaluate(model, X, y, batch_size)
    
    # Convert indices to cell type labels
    # Note: y_true and y_pred are indices into full_count_codes_pdf, not 'code' values
    # We need to use .iloc to access by position
    try:
        y_true_labels = [full_count_codes_pdf.iloc[idx]['cell_type'] for idx in y_true]
        y_pred_labels = [full_count_codes_pdf.iloc[idx]['cell_type'] for idx in y_pred]
    except IndexError as e:
        print(f"IndexError occurred: {e}")
        print(f"full_count_codes_pdf shape: {full_count_codes_pdf.shape}")
        print(f"Max index in y_true: {max(y_true) if len(y_true) > 0 else 'N/A'}")
        print(f"Max index in y_pred: {max(y_pred) if len(y_pred) > 0 else 'N/A'}")
        print("First few values in y_true:", y_true[:10])
        print("First few values in y_pred:", y_pred[:10])
        raise
    
    # Calculate hierarchical metrics
    hierarchical_metrics = calculate_hierarchical_f_score(y_true_labels, y_pred_labels, ontology_graph)
    
    # Combine all metrics
    all_metrics = {**standard_metrics, **hierarchical_metrics}
    
    return all_metrics, y_true, all_preds, y_pred

## Load Test Data

Load the same test data as in the original notebook.

In [6]:
# Load cell type information
human_only_x10_cell_counts_pdf = pd.read_parquet(data_dir / "cellxgene" / "human_only_x10_cell_counts.parquet")
cell_types = list(human_only_x10_cell_counts_pdf.columns)

# Create cell type code mappings
full_count_codes = human_only_x10_cell_counts_pdf.columns[human_only_x10_cell_counts_pdf.sum() > 10000].to_series().astype(
    pd.CategoricalDtype(categories=cell_types)
).cat.codes
full_count_codes_pdf = full_count_codes.reset_index().rename(columns={"index": "cell_type", 0: "code"})

# Load test data
from src.embeddings import load_data_set

val_120k_pdf = load_data_set(data_dir / "cellxgene_embeddings" / "test_v1", "cell_type")

def get_X_y(pdf):
    embedding_cols = [col for col in pdf.columns if col.isdigit()]
    y = pdf["cell_type"].astype(pd.CategoricalDtype(categories=cell_types)).cat.codes
    full_count_indexer = y.isin(full_count_codes_pdf.code)
    X = pdf[full_count_indexer][embedding_cols].to_numpy()
    y_mapped = pd.merge(
        y[full_count_indexer].to_frame('cell_type_code'),
        full_count_codes_pdf.reset_index(),
        left_on="cell_type_code",
        right_on="code",
        how="left"
    )['index'].values
    return X, y_mapped

X_val_120k, y_val_120k = get_X_y(val_120k_pdf)
print(f"Test set shape: X={X_val_120k.shape}, y={y_val_120k.shape}")

Test set shape: X=(111784, 3072), y=(111784,)


In [7]:
test_example_pdf = pd.read_parquet(
  list((data_dir / "cellxgene_embeddings" / "test_v1").glob("*.parquet"))[0]
)

In [8]:
for col in test_example_pdf.columns:
  if not col.isdigit():
    print(col)

cell_type
assay
organism
kit
organism_ontology_term_id
tissue_ontology_term_id
tissue_type
assay_ontology_term_id
disease_ontology_term_id
cell_type_ontology_term_id
self_reported_ethnicity_ontology_term_id
development_stage_ontology_term_id
sex_ontology_term_id
donor_id
suspension_type
predicted_celltype
is_primary_data
disease
sex
tissue
self_reported_ethnicity
development_stage
observation_joinid


## Load Trained Model

Load the checkpoint from the original training.

In [9]:
import os
import wandb

wandb.login(key=os.getenv("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/rj/.netrc
wandb: Currently logged in as: honicky to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [10]:
# Method 2: Try alternative W&B API approach - Download both checkpoint and config
# W&B run details
wandb_entity = "honicky"
wandb_project = "cellxgene-mlp-v1"
wandb_run_id = "nwhis8xb"
checkpoint_filename = "mlp_checkpoint_6000.pt"
config_filename = "config.yaml"

try:
  print("Trying alternative W&B API method...")
  api = wandb.Api()
  
  # Try to get the run and file differently
  runs = api.runs(f"{wandb_entity}/{wandb_project}")
  target_run = None
  
  for run in runs:
    if run.id == wandb_run_id:
      target_run = run
      break
  
  if target_run:
    print(f"Found run: {target_run.name}")
    files = target_run.files()
    
    # Look for both checkpoint and config files
    checkpoint_file = None
    config_file = None
    
    for file in files:
      if checkpoint_filename in file.name:
        checkpoint_file = file
      elif config_filename in file.name:
        config_file = file
      
      # Break early if we found both files
      if checkpoint_file and config_file:
        break
    
    # Download checkpoint file
    if checkpoint_file:
      print(f"Found checkpoint file: {checkpoint_file.name}")
      checkpoint_path = Path(checkpoint_file.download(replace=True).name)
      print(f"✓ Downloaded checkpoint to: {checkpoint_path}")
    else:
      print(f"Checkpoint file {checkpoint_filename} not found in run files")
      checkpoint_path = None
    
    # Download config file
    if config_file:
      print(f"Found config file: {config_file.name}")
      config_path = Path(config_file.download(replace=True).name)
      print(f"✓ Downloaded config to: {config_path}")
    else:
      print(f"Config file {config_filename} not found in run files")
      config_path = None
      
  else:
    print(f"Run {wandb_run_id} not found")
    checkpoint_path = None
    config_path = None
    
except Exception as e2:
  print(f"Alternative W&B API method failed: {e2}")
  checkpoint_path = None
  config_path = None

# Load config if available
if 'config_path' in locals() and config_path and config_path.exists():
  import yaml
  with open(config_path, 'r') as f:
    config = yaml.safe_load(f)
  print(f"Loaded config: {config}")
else:
  print("Config file not available, using default parameters")
  config = None

Trying alternative W&B API method...
Found run: cerulean-fog-89
Found checkpoint file: nwhis8xb/mlp_checkpoint_6000.pt
✓ Downloaded checkpoint to: nwhis8xb/mlp_checkpoint_6000.pt
Found config file: config.yaml
✓ Downloaded config to: config.yaml
Loaded config: {'_wandb': {'value': {'cli_version': '0.19.11', 'm': [], 'python_version': '3.12.8', 't': {'1': [1, 5, 35, 53, 55, 95, 105], '2': [1, 5, 35, 53, 55, 95, 105], '3': [2, 3, 16, 23, 55, 61], '4': '3.12.8', '5': '0.19.11', '8': [1, 5], '12': '0.19.11', '13': 'linux-x86_64'}, 'visualize': {'confusion_matrix': {'panel_config': {'fieldSettings': {'Actual': 'Actual', 'Predicted': 'Predicted', 'nPredictions': 'nPredictions'}, 'panelDefId': 'wandb/confusion_matrix/v1', 'stringSettings': {'title': 'Confusion Matrix Curve'}, 'transform': {'name': 'tableWithLeafColNames'}, 'userQuery': {'queryFields': [{'args': [{'name': 'runSets', 'value': '${runSets}'}], 'fields': [{'fields': [], 'name': 'id'}, {'fields': [], 'name': 'name'}, {'fields': [],

In [11]:
config

{'_wandb': {'value': {'cli_version': '0.19.11',
   'm': [],
   'python_version': '3.12.8',
   't': {'1': [1, 5, 35, 53, 55, 95, 105],
    '2': [1, 5, 35, 53, 55, 95, 105],
    '3': [2, 3, 16, 23, 55, 61],
    '4': '3.12.8',
    '5': '0.19.11',
    '8': [1, 5],
    '12': '0.19.11',
    '13': 'linux-x86_64'},
   'visualize': {'confusion_matrix': {'panel_config': {'fieldSettings': {'Actual': 'Actual',
       'Predicted': 'Predicted',
       'nPredictions': 'nPredictions'},
      'panelDefId': 'wandb/confusion_matrix/v1',
      'stringSettings': {'title': 'Confusion Matrix Curve'},
      'transform': {'name': 'tableWithLeafColNames'},
      'userQuery': {'queryFields': [{'args': [{'name': 'runSets',
           'value': '${runSets}'}],
         'fields': [{'fields': [], 'name': 'id'},
          {'fields': [], 'name': 'name'},
          {'fields': [], 'name': '_defaultColorIndex'},
          {'args': [{'name': 'tableKey', 'value': 'confusion_matrix_table'}],
           'fields': [],
        

In [12]:
# Model architecture from original notebook
def create_mlp(input_dim, num_classes, n_hidden_layers, dropout):
    layers = []
    dims = [int(input_dim + (num_classes - input_dim) * (i + 1) / (n_hidden_layers + 1)) 
            for i in range(n_hidden_layers)]
    prev_dim = input_dim
    for h_dim in dims:
        layers.append(nn.Linear(prev_dim, h_dim))
        layers.append(nn.BatchNorm1d(h_dim))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout))
        prev_dim = h_dim
    layers.append(nn.Linear(prev_dim, num_classes))
    return nn.Sequential(*layers)

# Load checkpoint (adjust path as needed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model parameters from best run
input_dim = config['input_dim']['value']
num_classes = config['num_classes']['value']
n_hidden_layers = config['n_hidden_layers']['value']
dropout = config['dropout']['value']

# Initialize and load model
model = create_mlp(input_dim, num_classes, n_hidden_layers, dropout).to(device)

if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print("Model loaded from checkpoint")
else:
    print(f"Warning: Checkpoint not found at {checkpoint_path}")
    print("Using randomly initialized model for demonstration")

model.eval()

Model loaded from checkpoint


Sequential(
  (0): Linear(in_features=500, out_features=469, bias=True)
  (1): BatchNorm1d(469, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU()
  (3): Dropout(p=0.053105164103764924, inplace=False)
  (4): Linear(in_features=469, out_features=438, bias=True)
  (5): BatchNorm1d(438, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (6): ReLU()
  (7): Dropout(p=0.053105164103764924, inplace=False)
  (8): Linear(in_features=438, out_features=407, bias=True)
  (9): BatchNorm1d(407, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (10): ReLU()
  (11): Dropout(p=0.053105164103764924, inplace=False)
  (12): Linear(in_features=407, out_features=377, bias=True)
)

## Build Cell Type Hierarchy

Create the ontology graph for hierarchical evaluation.

In [13]:
# Build cell type hierarchy graph
print("Building cell type hierarchy...")
ontology_graph = build_cell_type_graph(ontology, cell_types)
print(f"Graph has {len(ontology_graph.nodes)} nodes and {len(ontology_graph.edges)} edges")

# For cells not in ontology, we'll just use exact matching
# In practice, you'd want to map your cell types to ontology terms

Building cell type hierarchy...
Graph has 16863 nodes and 25581 edges


## Run Evaluation

Evaluate the model with both standard and hierarchical metrics.

In [14]:
print("Evaluating model...")
print("=" * 50)

# Evaluate with hierarchical metrics
all_metrics, y_true, all_preds, y_pred = evaluate_with_hierarchy(
    model, 
    X_val_120k[:, :input_dim],  # Use only first 500 dimensions
    y_val_120k,
    cell_types,
    full_count_codes_pdf,
    ontology_graph
)

# Display results
print("\nStandard Metrics:")
print("-" * 30)
for metric in ['logloss', 'macro_f1', 'macro_precision', 'macro_recall']:
    print(f"{metric:20s}: {all_metrics[metric]:.4f}")

print("\nRanking Metrics:")
print("-" * 30)
for k in [2, 5, 10]:
    print(f"Recall@{k:2d}: {all_metrics[f'recall_at_{k}']:.4f}  "
          f"MRR@{k:2d}: {all_metrics[f'mrr_at_{k}']:.4f}  "
          f"DCG@{k:2d}: {all_metrics[f'dcg_at_{k}']:.4f}")

print("\nHierarchical Metrics:")
print("-" * 30)
print(f"Hierarchical Precision: {all_metrics['hierarchical_precision']:.4f}")
print(f"Hierarchical Recall:    {all_metrics['hierarchical_recall']:.4f}")
print(f"Hierarchical F1:        {all_metrics['hierarchical_f1']:.4f}")

Evaluating model...

Standard Metrics:
------------------------------
logloss             : 2.1264
macro_f1            : 0.0918
macro_precision     : 0.1093
macro_recall        : 0.0924

Ranking Metrics:
------------------------------
Recall@ 2: 0.5617  MRR@ 2: 0.4894  DCG@ 2: 0.5084
Recall@ 5: 0.7633  MRR@ 5: 0.5460  DCG@ 5: 0.6002
Recall@10: 0.8704  MRR@10: 0.5605  DCG@10: 0.6350

Hierarchical Metrics:
------------------------------
Hierarchical Precision: 0.9127
Hierarchical Recall:    0.9030
Hierarchical F1:        0.9078


## Additional OnClass Integration

If OnClass is properly installed, we can use its built-in functions for more sophisticated analysis.

In [15]:
try:
    from OnClass.OnClassModel import OnClassModel
    from OnClass.OnClassUtils import load_cell_type_nlp_emb, load_cell_ontology
    
    print("OnClass successfully imported!")
    print("\nOnClass provides:")
    print("- Pre-trained models for Cell Ontology classification")
    print("- Graph-based embeddings of cell types")
    print("- Ability to predict unseen cell types")
    print("- Built-in hierarchical evaluation")
    
    # Example: Load OnClass pre-trained embeddings
    # cell_type_nlp_emb = load_cell_type_nlp_emb()
    # print(f"Loaded embeddings for {len(cell_type_nlp_emb)} cell types")
    
except ImportError:
    print("OnClass not installed. To use OnClass features:")
    print("1. git clone https://github.com/wangshenguiuc/OnClass.git")
    print("2. cd OnClass && pip install .")
    print("3. Download pre-trained models from onclass.ds.czbiohub.org")

OnClass not installed. To use OnClass features:
1. git clone https://github.com/wangshenguiuc/OnClass.git
2. cd OnClass && pip install .
3. Download pre-trained models from onclass.ds.czbiohub.org
